# Execução do RAG no Google Colab

Este notebook foi preparado para rodar o sistema RAG via Colab utilizando a estrutura de pastas do projeto. Ele assume que você já tem a Base de Conhecimento (KB) montada e quer apenas realizar consultas e experimentos.

## 1. Conexão com o Ambiente e Dependências

In [ ]:
!pip install langchain langchain-huggingface langchain-community langchain-experimental langchain-core langchain-classic pydantic langchain-chroma chromadb sentence-transformers pypdf pymupdf python-dotenv rank-bm25 --quiet

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# AJUSTE ESTE CAMINHO para a pasta do seu projeto no Drive
project_path = '/content/drive/MyDrive/projeto-1-rag-rag-5-anos-em-50'
os.chdir(project_path)

print(f"Diretório atual: {os.getcwd()}")
print("Arquivos na pasta:", os.listdir())

import sys
import os

# Adiciona a pasta src ao path
sys.path.append(os.path.join(os.getcwd(), 'src'))

from rag import MyRAG
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import torch

## 2. Inicialização do Modelo (LLM)

Aqui instanciamos o modelo que será usado para gerar as respostas. No exemplo abaixo, usamos o `Gemma-2b-it` via HuggingFace.

In [ ]:
MODEL_ID = "google/gemma-3-4b-it"

local_llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs=dict(
        do_sample=True,
        max_new_tokens=2048,
        return_full_text=False
    )
)
chat_model = ChatHuggingFace(llm=local_llm)
rag = MyRAG(llm_instance=chat_model)

## 3. Realizando Perguntas (Experimentos)

Agora você pode testar o seu RAG com diferentes perguntas.

In [ ]:
question = """
    A trágica quebra institucional do golpe civil-militar de 1964 interrompeu a sobrevida política de JK, situação que atingiu o ápice com a dolorosa cassação do seu mandato de senador e suspensão severa de seus direitos políticos em 8 de junho de 1964. Como o autor descreve detalhadamente a mecânica de pressão política e militar que emparedou o governo de Humberto Castello Branco e forçou a punição sumária do ex-presidente?
    A) O presidente Castello Branco, motivado intimamente por um repúdio ideológico cultivado durante toda a sua carreira, exigiu e redigiu o decreto de expurgo de forma autoritária e despótica, isolando-se das deliberações conjuntas do Conselho de Segurança Nacional Agindo por vontade unânime própria, o general argumentou, baseando-se em convicções irrevogáveis e anunciando publicamente na TV, que Juscelino era o orquestrador financeiro da guerrilha comunista que tomou conta das praças de Belo Horizonte.
    B) O afastamento compulsório e dramático de Juscelino foi exigido de forma quase intempestiva e veemente pelo ministro da Guerra, o general Costa e Silva, atuando sob forte clamor dos oficiais da "linha dura" e lacerdistas, os quais consideravam o senador o pior e maior entrave oculto da Revolução em decorrência de seu formidável favoritismo e projeção para as futuras eleições presidenciais. Pressionado impiedosamente por esse flanco radical militar e confrontado por um dossiê calhamaço com suspeitas insatisfatórias de corrupção pessoal, o ponderado Castello Branco cedeu à contingência e executou burocraticamente a condenação.
    C) Após conversações diplomáticas madrugadoras travadas discretamente num requintado apartamento em Copacabana, Juscelino optou por solicitar oficialmente ao general Castello Branco a suspensão abrandada dos seus próprios direitos eleitorais. Ao prever um insustentável derramamento de sangue que atingiria a oposição de sua época, JK aceitou migrar num exílio protegido até Paris, exigindo como única moeda de troca para o abandono político o direito de seu PSD figurar oficialmente na estruturação legal e militar do novo governo instituído e assumir postos nos ministérios da Defesa.
    D) Uma vigorosa articulação originada dentro dos gabinetes do Supremo Tribunal Federal, sob influência das intensas e assustadoras passeatas que cobravam probidade no país, elaborou e aplicou uma decisão unânime limitadora do mandato e imunidade de Juscelino. O poder Judiciário considerou devidamente provadas todas as transações escusas do político mineiro com embaixadores ideológicos em Havana, coagindo moral e legalmente um presidente Castello Branco hesitante a endossar o afastamento do líder como uma determinação inquestionável e constitucional da Justiça pátria.
    """

resposta, chunks = rag.answer_question(question, mostrar_chunks=True)

print("\n--- RESPOSTA DO RAG ---\n")
print(resposta)

In [ ]:
# Teste com outra pergunta
pergunta_2 = "Qual foi o papel do general Costa e Silva no afastamento de Juscelino?"
res_2 = rag.answer_question(pergunta_2)
print(res_2)

## 4. Finalização

Liberando os recursos.

In [ ]:
rag.teardown()
del rag
torch.cuda.empty_cache()